# Thực nghiệm 1: CP Decomposition trên dữ liệu EEG

Trong notebook này, chúng ta sẽ thực hiện phân rã tensor CP trên dữ liệu EEG thực tế từ bộ dataset PhysioNet EEG Motor Movement/Imagery. Tensor có 3 chiều: **Subjects $\times$ Channels $\times$ Time**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data import load_eeg_raw, build_tensor, get_channel_positions
from src.cp_als import cp_als
from src.evaluate import evaluate_ranks
from src.visualize import plot_eeg_components, plot_reconstruction_error

# Tắt warnings của MNE cho output gọn gàng
import mne
mne.set_log_level('WARNING')

## 1. Tải và Xây dựng Tensor từ Dữ liệu EEG

Chúng ta sẽ sử dụng 20 subjects, lấy các run liên quan đến motor imagery (left fist vs right fist) là run 4, 8, 12.

In [ ]:
n_subjects = 20
subjects = list(range(1, n_subjects + 1))
runs = [4, 8, 12]

print("Đang tải và tiền xử lý dữ liệu EEG...")
raws = load_eeg_raw(subjects, runs, verbose=False)

# Xây dựng tensor: Lọc băng thông 8-30Hz (Mu và Beta), lấy epochs từ -0.5s đến 2.5s
tensor, subj_ids, info = build_tensor(
    raws, 
    l_freq=8.0, h_freq=30.0, 
    tmin=-0.5, tmax=2.5,
    average_trials=True, 
    verbose=True
)

## 2. Phân rã Tensor với CP-ALS

Đầu tiên, chúng ta sẽ đánh giá nhiều mức rank khác nhau để tìm ra rank phù hợp (sử dụng CORCONDIA và Reconstruction Error). Tuy nhiên, để tiết kiệm thời gian minh họa, chúng ta sẽ chạy CP-ALS với rank = 5.

In [ ]:
R = 5
print(f"Chạy CP-ALS với rank = {R}...")
weights, f0, f1, f2 = cp_als(tensor, rank=R, max_iter=200, tol=1e-6, verbose=True)
factors = [f0, f1, f2]

## 3. Lựa chọn Rank (Rank Selection)

Để thực nghiệm nghiêm túc, chúng ta nên quét qua các rank từ 2 đến 10. (Lưu ý: Quá trình này có thể mất vài phút).

In [ ]:
# ranks_to_test = [2, 3, 4, 5, 6, 7, 8]
# print("Đánh giá các mức rank...")
# results = evaluate_ranks(tensor, ranks_to_test, cp_als, max_iter=100)
# plot_reconstruction_error(results['ranks'], results['errors'], results['corcondia'])
# plt.show()

print("Đã comment phần quét rank để chạy nhanh hơn trong lần đầu.")

## 4. Trực quan hóa và Diễn giải (Interpretation)

Vẽ các thành phần nhận được từ CP-ALS. Mỗi thành phần đại diện cho một mẫu hoạt động não (Spatial Pattern) đi kèm với động học thời gian (Temporal Pattern) và sự phân bố trên các subject (Subject Pattern).

In [ ]:
channel_names = info['channel_names']
times = info['times']

plot_eeg_components(factors, channel_names, times=times, sfreq=info['sfreq'], weights=weights)
plt.show()

**Giải thích (Ví dụ mong đợi)**:
- Nếu chúng ta thấy một component tập trung ở C3/C4 (vùng vận động) và có sự thay đổi biên độ rõ rệt quanh thời điểm $t=0$, đó chính là tín hiệu ERD/ERS đặc trưng của Motor Imagery.
- Component ở vùng chẩm (O1/O2) có thể liên quan đến Alpha rhythm (visual cortex).
- Component ở vùng trán (Fp1/Fp2) thường do nhiễu chớp mắt (EOG artifact).